In [1]:
import sys
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate
sys.path.insert(1, r'C:\Users\Dharmendra Vartika\LLM\env')
from enviorment import load_env
import os 
load_env()
model =ChatOpenAI()
from langgraph.graph import StateGraph,START, END
from typing import TypedDict


In [2]:
class LMMstate(TypedDict):
    topic :str
    outline: str
    blog :str

In [3]:
parser=StrOutputParser()

In [5]:
def create_outline (state: LMMstate)->LMMstate:
    topic =state['topic']
    promt =PromptTemplate(template= 'create a outline on the topic \n {topic}',
                          input_variables=['topic'])
    chain =promt | model | parser
    state['outline']=chain.invoke ({'topic':topic})
    return state


In [6]:
def create_blog (state: LMMstate)->LMMstate:
    topic =state['topic']
    outline =state['outline']
    promt =PromptTemplate(template= 'using this outline and topic please create 50 lines blog \n {topic} {outline}',
                          input_variables=['topic','outline'])
    chain =promt | model | parser
    state['blog']=chain.invoke ({'topic':topic,'outline':outline})
    return state


In [7]:
graph=StateGraph(LMMstate)
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)

In [9]:
workflow=graph.compile()

In [11]:
state ={'topic':'Cricket'}
workflow.invoke(state)

{'topic': 'Cricket',
 'outline': 'I. Introduction \n    A. Brief history of cricket \n    B. Popularity of cricket worldwide \n\nII. Rules and Gameplay \n    A. Equipment used in cricket \n    B. Objectives of the game \n    C. Overview of the rules and structure of a cricket match \n    D. Types of cricket matches: Test matches, One Day Internationals, Twenty20 \n\nIII. Players and Positions \n    A. Roles and responsibilities of different players on a cricket team \n    B. Key positions on the field: batsmen, bowlers, wicket-keeper \n    C. Importance of teamwork in cricket \n\nIV. Major Tournaments and Events \n    A. Overview of international cricket tournaments: ICC Cricket World Cup, ICC Champions Trophy \n    B. Importance of domestic cricket leagues \n    C. Impact of cricket on local and global communities \n\nV. Evolution and Future of Cricket \n    A. Changes and innovations in cricket over the years \n    B. Challenges facing the sport \n    C. Opportunities for growth and 